# BB84 Teaching Trace

This copy is for watching the quantum part of BB84 move through time.

We keep the real event pipeline: Alice source, quantum channel, Bob detector, classical `quantum.done`, Bob basis announcement, and Alice accepted sifted slots. The parameters are intentionally small and slightly non-ideal so the trace shows emissions, losses, detections, and sifting without producing thousands of lines.


In [ ]:
from dataclasses import dataclass
import math
from simyuj.components import GaussianTiming, SinglePhotonSource
from simyuj.primitives.units import seconds_to_ticks
from simyuj.qstate import StateSampler
from simyuj.signal import EncodingScheme
from simyuj.components import QuantumChannel
from simyuj.qstate.noise import depolarizing
from simyuj.components.detectors import (
    ACTION_DETECT_SIGNAL,
    DetectorArray,
    Measure,
    SinglePhotonDetector,
    SinglePhotonDetectorParams,
)
from simyuj.components.detectors.primitives.click import ThresholdClickResolver
from dataclasses import field
from simyuj.components.detectors import DetectionReport
from simyuj.components import SourcePreparationReport
from simyuj.control import AGENT_REPORT, NodeAgent
from simyuj.components import ACTION_TRANSMIT_QUANTUM
from simyuj.engine import Timeline
from simyuj.network import Network, Node
from simyuj.control import SessionRuntime
from pathlib import Path
import sys

REPO_ROOT = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "simyuj").exists() and (candidate / "examples").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not find the simyuj repository root")

PROTOCOLS_DIR = REPO_ROOT / "tutorials" / "protocols"
PROTOCOLS_DIR.mkdir(parents=True, exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from examples.postprocessing.bb84_event.messages import (
    encode_body,
    decode_body,
)
from examples.postprocessing.bb84_event.helpers import (
    binary_entropy,
    choose_sample_positions,
    parity,
    random_bits,
    remove_positions,
    require_binary_bits,
    require_positions,
    toeplitz_hash,
)
from examples.postprocessing.bb84_event.cascade import CascadeController


from simyuj.tracing.levels import LogLevel
from simyuj.tracing.logger import SimulationLogger
from simyuj.tracing.sinks import JsonlSink
from simyuj.components import ACTION_TRANSMIT_CLASSICAL, ClassicalChannel
from simyuj.control import AGENT_MESSAGE
import json

from simyuj.control import AGENT_EVENT
from simyuj.engine import Event
from simyuj.primitives.messages import ClassicalMessage


In [ ]:
# bb84 states 

BB84_STATES = ("|0>", "|1>", "|+>", "|->")
BB84_LABELS = ("Z0", "Z1", "X0", "X1")


# helpers


In [ ]:
def bb84_basis_bit(label: str) -> tuple[str, int]:
    if label not in BB84_LABELS:
        raise ValueError(f"unknown BB84 label: {label!r}")
    return label[0], int(label[1])


def bob_report_basis_bit(report) -> tuple[str, int] | None:
    if not report.success:
        return None

    basis = report.measurement_label
    outcome = report.outcome

    if basis == "Z" and outcome == "0":
        return "Z", 0
    if basis == "Z" and outcome == "1":
        return "Z", 1
    if basis == "X" and outcome == "+":
        return "X", 0
    if basis == "X" and outcome == "-":
        return "X", 1

    raise ValueError(
        f"unexpected Bob report outcome: basis={basis!r}, outcome={outcome!r}"
    )


def send_json_message(
    agent,
    ctx,
    *,
    receiver_id: str,
    out_port: str,
    message_type: str,
    body: dict,
):
    message = ClassicalMessage(
        sender_id=agent.agent_id,
        receiver_id=receiver_id,
        body=encode_body(body),
        sent_time=ctx.timeline.current_time,
        session_id=ctx.session_id,
        message_type=message_type,
        message_id=f"{agent.agent_id}-{agent.message_counter}",
    )
    agent.message_counter += 1
    return agent.classical.send(
        message,
        ctx.timeline,
        port_name=out_port,
    )


def bb84_final_key_length(
    *,
    reconciled_len: int,
    estimated_qber: float,
    revealed_bits: int,
    qber_safety_margin: float,
    security_margin_bits: int,
) -> int:
    qber_for_pa = min(0.5, max(0.0, estimated_qber + qber_safety_margin))
    phase_error_cost = math.ceil(reconciled_len * binary_entropy(qber_for_pa))
    final_len = (
        reconciled_len
        - phase_error_cost
        - revealed_bits
        - security_margin_bits
    )
    return max(0, final_len)


def bb84_source_done_delay_ticks(source_config) -> int:
    frame_duration_s = source_config.num_slots / source_config.clock_hz
    source_margin_s = source_config.max_timing_jitter_s
    return seconds_to_ticks(frame_duration_s + source_margin_s)


def bb84_bob_sifting_guard_ticks(detector_config) -> int:
    guard_s = (
        6 * detector_config.jitter_stddev_s
        + detector_config.detection_window_s
        + detector_config.output_latency_s
    )
    return seconds_to_ticks(guard_s)


def bb84_slot_period_ticks(source_config) -> int:
    return seconds_to_ticks(1.0 / source_config.clock_hz)


def bb84_slot_zero_arrival_tick(quantum_config) -> int:
    return seconds_to_ticks(
        quantum_config.length_m / quantum_config.propagation_speed_m_per_s
    )


def bb84_slot_assignment_window_ticks(
    source_config,
    quantum_config,
    detector_config,
) -> int:
    window_s = (
        source_config.max_timing_jitter_s
        + 6 * quantum_config.timing_jitter_stddev_s
        + 6 * detector_config.jitter_stddev_s
        + detector_config.detection_window_s
    )
    return seconds_to_ticks(window_s)


def assign_detection_slot_index(
    detection_time: int,
    *,
    slot_period_ticks: int,
    slot_zero_arrival_tick: int,
    slot_assignment_window_ticks: int,
    num_slots: int,
) -> int | None:
    if slot_period_ticks <= 0:
        raise ValueError("slot_period_ticks must be positive")

    relative_tick = detection_time - slot_zero_arrival_tick
    slot_offset = round(relative_tick / slot_period_ticks)
    slot_index = slot_offset + 1

    if slot_index < 1 or slot_index > num_slots:
        return None

    expected_tick = slot_zero_arrival_tick + slot_offset * slot_period_ticks
    if abs(detection_time - expected_tick) > slot_assignment_window_ticks:
        return None

    return slot_index


def report_detection_time(report) -> int:
    if report.raw_clicks:
        return min(click.time for click in report.raw_clicks)
    return report.time


# device configs


In [ ]:
@dataclass(frozen=True)
class BB84AliceSourceConfig:
    device_id: str = "alice_source"
    
    clock_hz: float = 100e6

    # Number of emission slots to simulate.
    num_slots: int = 20

    emission_probability: float = 1.0

    wavelength_nm: float = 1550.0

    # Source timing jitter.
    timing_jitter_stddev_s: float = 20e-12
    max_timing_jitter_s: float = 100e-12


@dataclass(frozen=True)
class BB84QuantumChannelConfig:
    channel_id: str = "alice_to_bob_quantum"

    length_m: float = 25_000

    attenuation_db_per_km: float = 0.20

    fixed_insertion_loss_db: float = 2.0

    propagation_speed_m_per_s: float = 2.0e8

    # Channel timing jitter.
    timing_jitter_stddev_s: float = 50e-12

    depolarizing_probability: float = 0.02


@dataclass(frozen=True)
class BB84ClassicalChannelConfig:
    length_m: float = 25_000
    fiber_speed_m_per_s: float = 2.0e8

    # authenticated public channel should be reliable for now.
    # We can model message loss later but post-processing is easier to debug first.
    loss_probability: float = 0.0


@dataclass(frozen=True)
class BB84BobDetectorConfig:
    device_id: str = "bob_detector"

    efficiency: float = 0.65
    dark_count_rate_hz: float = 100.0

    dead_time_s: float = 50e-9

    jitter_stddev_s: float = 50e-12

    p_afterpulse: float = 0.001
    afterpulse_decay_s: float = 100e-9

    detection_window_s: float = 500e-12

    output_latency_s: float = 5e-9


In [ ]:
@dataclass(frozen=True)
class BB84QBERConfig:
    sample_fraction: float = 0.20
    min_sample_bits: int = 16
    min_remaining_bits: int = 32
    abort_threshold: float = 0.11


qber_config = BB84QBERConfig()


In [ ]:
@dataclass(frozen=True)
class BB84CascadeConfig:
    passes: int = 4
    min_block_size: int = 2

    # Used only to choose Cascade block sizes.
    # This prevents QBER=0 sample from making blocks too large.
    block_qber_floor: float = 0.02
    max_first_block_size: int = 32


cascade_config = BB84CascadeConfig()


In [ ]:
@dataclass(frozen=True)
class BB84VerificationConfig:
    # Failure probability is about 2^-tag_len if the reconciled keys differ.
    tag_len: int = 64


verification_config = BB84VerificationConfig()


In [ ]:
@dataclass(frozen=True)
class BB84PrivacyAmplificationConfig:
    # Conservative cushion because QBER was estimated from a sample.
    qber_safety_margin: float = 0.02

    # Security slack for this notebook-level finite-key model.
    security_margin_bits: int = 64

    # Abort if the extracted key would be too small to be meaningful.
    min_final_key_bits: int = 32


privacy_config = BB84PrivacyAmplificationConfig()


# alices agent


In [ ]:
@dataclass(frozen=True)
class AlicePreparationRecord:
    slot_index: int
    signal_id: str
    time: int
    emission_slot_tick: int
    emission_delay_ticks: int
    attempt_index: int
    emission_index: int
    basis: str
    bit: int
    sampler_label: str
    report_id: str


In [ ]:
@dataclass(slots=True)
class BB84AliceAgent(NodeAgent):
    source: SinglePhotonSource
    quantum_done_delay_ticks: int = 0

    preparations: list[AlicePreparationRecord] = field(default_factory=list)
    preparations_by_signal_id: dict[str, AlicePreparationRecord] = field(
        default_factory=dict
    )
    preparations_by_slot_index: dict[int, AlicePreparationRecord] = field(
        default_factory=dict
    )

    peer_id: str = "bob_agent"
    out_port: str = "to_bob"
    sifted_slot_indices: list[int] = field(default_factory=list)
    sifted_signal_ids: list[str] = field(default_factory=list)
    sifted_bits: list[int] = field(default_factory=list)
    no_emission_sift_rejections: int = 0
    sifting_complete: bool = False

    sample_positions: list[int] = field(default_factory=list)
    estimated_qber: float | None = None
    sample_errors: int = 0
    qber_accepted: bool | None = None
    reconciled_bits: list[int] = field(default_factory=list)
    qber_complete: bool = False
    aborted_reason: str | None = None

    cascade_parity_requests: int = 0
    cascade_leaked_bits: int = 0

    verification_complete: bool = False
    verification_accepted: bool | None = None
    verification_leaked_bits: int = 0

    privacy_complete: bool = False
    final_key: list[int] = field(default_factory=list)
    final_key_length: int = 0
    privacy_revealed_bits: int = 0

    message_counter: int = 0
    _sample_rng: object | None = field(default=None, repr=False)

    def bind(self, context) -> None:
        self._sample_rng = context.timeline.rng(
            self.agent_id,
            "bb84",
            "qber_sample",
        )

    def _schedule_local_action(
        self,
        ctx,
        action: str,
        *,
        delay_ticks: int = 0,
    ) -> None:
        ctx.timeline.schedule(
            Event(
                time=ctx.timeline.current_time + delay_ticks,
                target_ref=self,
                action=AGENT_EVENT,
                payload_ref={"bb84_action": action},
                source=self,
                subsystem_id="control",
                meta={
                    "session_id": ctx.session_id,
                    "agent_id": self.agent_id,
                    "bb84_action": action,
                },
            )
        )

    def on_start(self, start, ctx) -> None:
        del start
        self.source.schedule_start(ctx.timeline)
        self._schedule_local_action(
            ctx,
            "quantum.done",
            delay_ticks=self.quantum_done_delay_ticks,
        )

    def on_report(self, report: object, ctx) -> None:
        del ctx

        if not isinstance(report, SourcePreparationReport):
            raise TypeError(f"Alice received unsupported report: {type(report)!r}")

        if report.sampler_label is None:
            raise ValueError("source report must include a BB84 sampler label")

        basis, bit = bb84_basis_bit(report.sampler_label)

        for signal_id in report.signal_ids:
            record = AlicePreparationRecord(
                slot_index=int(report.attempt_index),
                signal_id=signal_id,
                time=report.time,
                emission_slot_tick=report.emission_slot_tick,
                emission_delay_ticks=report.emission_delay_ticks,
                attempt_index=report.attempt_index,
                emission_index=report.emission_index,
                basis=basis,
                bit=bit,
                sampler_label=report.sampler_label,
                report_id=report.report_id,
            )
            self.preparations.append(record)
            self.preparations_by_signal_id[signal_id] = record
            self.preparations_by_slot_index[record.slot_index] = record

    def on_message(self, message, ctx) -> None:
        message_type = message.message.message_type

        if message_type == "sift.bob_bases":
            self._on_bob_bases(message, ctx)
            return

        if message_type == "estimate.result":
            self._on_estimate_result(message, ctx)
            return

        if message_type == "cascade.parity_request":
            self._on_cascade_parity_request(message, ctx)
            return
        
        if message_type == "verify.tag":
            self._on_verify_tag(message, ctx)
            return
        
        if message_type == "privacy.seed":
            self._on_privacy_seed(message, ctx)
            return

        raise ValueError(f"unsupported Alice message: {message_type!r}")

    def on_event(self, event: Event, ctx) -> None:
        payload = event.payload_ref
        if not isinstance(payload, dict):
            raise TypeError("Alice local event payload must be dict")

        action = payload.get("bb84_action")

        if action == "quantum.done":
            self._send_quantum_done(ctx)
            return

        raise ValueError(f"unsupported Alice local action: {payload!r}")

    def _send_quantum_done(self, ctx) -> None:
        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="quantum.done",
            body={
                "prepared_count": len(self.preparations),
                "done_time": ctx.timeline.current_time,
            },
        )

    def _on_bob_bases(self, message, ctx) -> None:
        body = decode_body(message)
        detections = body["detections"]

        accepted_slot_indices = []
        seen_slot_indices = set()
        self.no_emission_sift_rejections = 0

        for item in detections:
            slot_index = int(item["slot_index"])
            bob_basis = str(item["basis"])

            if bob_basis not in ("Z", "X"):
                raise ValueError(f"invalid Bob basis: {bob_basis!r}")

            if slot_index in seen_slot_indices:
                raise ValueError(f"duplicate Bob slot_index announcement: {slot_index}")

            seen_slot_indices.add(slot_index)

            alice_record = self.preparations_by_slot_index.get(slot_index)
            if alice_record is None:
                self.no_emission_sift_rejections += 1
                continue

            if alice_record.basis == bob_basis:
                accepted_slot_indices.append(slot_index)

        self.sifted_slot_indices = accepted_slot_indices
        self.sifted_signal_ids = [
            self.preparations_by_slot_index[slot_index].signal_id
            for slot_index in accepted_slot_indices
        ]
        self.sifted_bits = [
            self.preparations_by_slot_index[slot_index].bit
            for slot_index in accepted_slot_indices
        ]
        self.reconciled_bits = self.sifted_bits.copy()
        self.sifting_complete = True

        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="sift.accepted",
            body={"slot_indices": accepted_slot_indices},
        )

        self._start_qber_estimation(ctx)

    def _start_qber_estimation(self, ctx) -> None:
        if not self.sifting_complete:
            raise RuntimeError("cannot estimate QBER before sifting completes")

        n = len(self.sifted_bits)
        needed = qber_config.min_sample_bits + qber_config.min_remaining_bits
        if n < needed:
            self.aborted_reason = (
                f"not enough sifted bits for QBER estimation: have {n}, need {needed}"
            )
            send_json_message(
                self,
                ctx,
                receiver_id=self.peer_id,
                out_port=self.out_port,
                message_type="abort",
                body={"reason": self.aborted_reason},
            )
            return

        if self._sample_rng is None:
            raise RuntimeError("Alice QBER sample RNG is not bound")

        sample_size = min(
            n - qber_config.min_remaining_bits,
            max(qber_config.min_sample_bits, int(qber_config.sample_fraction * n)),
        )

        self.sample_positions = choose_sample_positions(
            self._sample_rng,
            n,
            sample_size,
        )
        sample_bits = [
            self.sifted_bits[position]
            for position in self.sample_positions
        ]

        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="estimate.sample",
            body={
                "sample_positions": self.sample_positions,
                "sample_bits": sample_bits,
            },
        )

    def _on_estimate_result(self, message, ctx) -> None:
        del ctx

        body = decode_body(message)

        self.estimated_qber = float(body["qber"])
        self.qber_accepted = bool(body["accept"])
        self.sample_errors = int(body["sample_errors"])

        returned_positions = [int(pos) for pos in body["sample_positions"]]
        require_positions(
            "sample_positions",
            returned_positions,
            len(self.sifted_bits),
        )

        if returned_positions != self.sample_positions:
            raise ValueError("Bob returned different sample positions")

        self.reconciled_bits = remove_positions(
            self.sifted_bits,
            self.sample_positions,
        )
        self.qber_complete = True

        if not self.qber_accepted:
            self.aborted_reason = "estimated QBER exceeds abort threshold"

    def _on_cascade_parity_request(self, message, ctx) -> None:
        if not self.qber_complete:
            raise RuntimeError("cannot answer Cascade before QBER estimation completes")

        body = decode_body(message)
        request_id = str(body["request_id"])
        indices = [int(index) for index in body["indices"]]

        require_positions(
            "cascade request indices",
            indices,
            len(self.reconciled_bits),
        )

        self.cascade_parity_requests += 1
        self.cascade_leaked_bits += 1

        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="cascade.parity_response",
            body={
                "request_id": request_id,
                "parity": parity(self.reconciled_bits, indices),
            },
        )

    def _on_verify_tag(self, message, ctx) -> None:
        if not self.qber_complete:
            raise RuntimeError("cannot verify before QBER estimation completes")

        body = decode_body(message)

        input_len = int(body["input_len"])
        tag_len = int(body["tag_len"])
        tag_seed = [int(bit) for bit in body["tag_seed"]]
        bob_tag = [int(bit) for bit in body["tag"]]

        require_binary_bits("verification tag_seed", tag_seed)
        require_binary_bits("verification bob_tag", bob_tag)

        verified = False

        if input_len == len(self.reconciled_bits):
            alice_tag = toeplitz_hash(
                self.reconciled_bits,
                tag_seed,
                tag_len,
            )
            verified = alice_tag == bob_tag

        self.verification_complete = True
        self.verification_accepted = verified
        self.verification_leaked_bits = tag_len

        if not verified:
            self.aborted_reason = "verification failed"

        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="verify.result",
            body={
                "verified": verified,
                "tag_len": tag_len,
            },
        )

    def _on_privacy_seed(self, message, ctx) -> None:
        if not self.verification_accepted:
            raise RuntimeError("cannot run privacy amplification before verification passes")

        body = decode_body(message)

        input_len = int(body["input_len"])
        final_key_len = int(body["final_key_len"])
        seed = [int(bit) for bit in body["seed"]]

        require_binary_bits("privacy seed", seed)

        if input_len != len(self.reconciled_bits):
            raise ValueError("privacy amplification input length mismatch")

        if final_key_len < privacy_config.min_final_key_bits:
            self.aborted_reason = "final key too short"
            return

        self.final_key = toeplitz_hash(
            self.reconciled_bits,
            seed,
            final_key_len,
        )
        self.final_key_length = len(self.final_key)
        self.privacy_revealed_bits = int(body["revealed_bits"])
        self.privacy_complete = True

        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="privacy.done",
            body={
                "final_key_len": self.final_key_length,
            },
        )


# bobs agent


In [ ]:
@dataclass(frozen=True)
class BobDetectionRecord:
    slot_index: int
    signal_id: str
    time: int
    detection_time: int
    basis: str
    bit: int
    outcome: object
    report_id: str
    detector_id: str | None
    flags: tuple[str, ...]


In [ ]:
@dataclass(slots=True)
class BB84BobAgent(NodeAgent):
    sifting_guard_ticks: int = 0
    slot_period_ticks: int = 0
    slot_zero_arrival_tick: int = 0
    slot_assignment_window_ticks: int = 0
    num_slots: int = 0

    reports: list[DetectionReport] = field(default_factory=list)
    detections: list[BobDetectionRecord] = field(default_factory=list)
    detections_by_signal_id: dict[str, BobDetectionRecord] = field(
        default_factory=dict
    )
    detections_by_slot_index: dict[int, BobDetectionRecord] = field(
        default_factory=dict
    )
    failed_reports: list[DetectionReport] = field(default_factory=list)
    unassigned_reports: list[DetectionReport] = field(default_factory=list)
    duplicate_slot_reports: list[DetectionReport] = field(default_factory=list)

    quantum_done_received: bool = False
    quantum_done_time: int | None = None
    sifting_started: bool = False

    peer_id: str = "alice_agent"
    out_port: str = "to_alice"
    sifted_slot_indices: list[int] = field(default_factory=list)
    sifted_signal_ids: list[str] = field(default_factory=list)
    sifted_bits: list[int] = field(default_factory=list)
    sifting_complete: bool = False

    sample_positions: list[int] = field(default_factory=list)
    sample_errors: int = 0
    estimated_qber: float | None = None
    qber_accepted: bool | None = None
    reconciled_bits: list[int] = field(default_factory=list)
    qber_complete: bool = False
    aborted_reason: str | None = None

    cascade_complete: bool = False
    cascade_first_block_size: int | None = None
    cascade_parity_requests: int = 0
    cascade_corrections: int = 0
    cascade_leaked_bits: int = 0
    _cascade_controller: CascadeController | None = field(default=None, repr=False)
    _cascade_rng: object | None = field(default=None, repr=False)

    verification_complete: bool = False
    verification_accepted: bool | None = None
    verification_tag_len: int = 0
    verification_leaked_bits: int = 0
    _verification_rng: object | None = field(default=None, repr=False)

    privacy_complete: bool = False
    final_key: list[int] = field(default_factory=list)
    final_key_length: int = 0
    privacy_revealed_bits: int = 0
    _privacy_rng: object | None = field(default=None, repr=False)

    message_counter: int = 0

    def bind(self, context) -> None:
        self._cascade_rng = context.timeline.rng(
            self.agent_id,
            "bb84",
            "cascade",
        )
        self._verification_rng = context.timeline.rng(
            self.agent_id,
            "bb84",
            "verification",
        )
        self._privacy_rng = context.timeline.rng(
            self.agent_id,
            "bb84",
            "privacy_amplification",
        )

    def on_start(self, start, ctx) -> None:
        del start, ctx

    def on_report(self, report: object, ctx) -> None:
        del ctx

        if not isinstance(report, DetectionReport):
            raise TypeError(f"Bob received unsupported report: {type(report)!r}")

        self.reports.append(report)

        decoded = bob_report_basis_bit(report)
        if decoded is None:
            self.failed_reports.append(report)
            return

        if report.signal_id is None:
            raise ValueError("successful Bob detection report must include signal_id")

        detection_time = report_detection_time(report)
        slot_index = assign_detection_slot_index(
            detection_time,
            slot_period_ticks=self.slot_period_ticks,
            slot_zero_arrival_tick=self.slot_zero_arrival_tick,
            slot_assignment_window_ticks=self.slot_assignment_window_ticks,
            num_slots=self.num_slots,
        )
        if slot_index is None:
            self.unassigned_reports.append(report)
            return

        if slot_index in self.detections_by_slot_index:
            self.duplicate_slot_reports.append(report)
            return

        basis, bit = decoded
        detector_id = report.raw_clicks[0].detector_id if report.raw_clicks else None

        record = BobDetectionRecord(
            slot_index=slot_index,
            signal_id=str(report.signal_id),
            time=report.time,
            detection_time=detection_time,
            basis=basis,
            bit=bit,
            outcome=report.outcome,
            report_id=report.report_id,
            detector_id=detector_id,
            flags=report.flags,
        )

        self.detections.append(record)
        self.detections_by_signal_id[record.signal_id] = record
        self.detections_by_slot_index[record.slot_index] = record

    def on_event(self, event: Event, ctx) -> None:
        payload = event.payload_ref
        if not isinstance(payload, dict):
            raise TypeError("Bob local event payload must be dict")

        action = payload.get("bb84_action")

        if action == "sifting.start":
            self._start_sifting(ctx)
            return

        if action == "cascade.next_request":
            self._cascade_next_request(ctx)
            return
        
        raise ValueError(f"unsupported Bob local action: {payload!r}")

    def on_message(self, message, ctx) -> None:
        message_type = message.message.message_type

        if message_type == "quantum.done":
            self._on_quantum_done(message, ctx)
            return

        if message_type == "sift.accepted":
            self._on_sift_accepted(message, ctx)
            return

        if message_type == "estimate.sample":
            self._on_estimate_sample(message, ctx)
            return

        if message_type == "cascade.parity_response":
            self._on_cascade_parity_response(message, ctx)
            return

        if message_type == "abort":
            body = decode_body(message)
            self.aborted_reason = str(body.get("reason", "peer aborted"))
            return
        
        if message_type == "verify.result":
            self._on_verify_result(message, ctx)
            return
        
        if message_type == "privacy.done":
            self._on_privacy_done(message, ctx)
            return

        raise ValueError(f"unsupported Bob message: {message_type!r}")

    def _on_quantum_done(self, message, ctx) -> None:
        body = decode_body(message)
        self.quantum_done_received = True
        self.quantum_done_time = int(body["done_time"])
        self._schedule_local_action(
            ctx,
            "sifting.start",
            delay_ticks=self.sifting_guard_ticks,
        )

    def _start_sifting(self, ctx) -> None:
        if self.sifting_started or self.sifting_complete:
            return

        self.sifting_started = True

        detections = [
            {"slot_index": record.slot_index, "basis": record.basis}
            for record in self.detections
        ]

        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="sift.bob_bases",
            body={"detections": detections},
        )

    def _on_sift_accepted(self, message, ctx) -> None:
        del ctx

        body = decode_body(message)
        accepted_slot_indices = [int(slot_index) for slot_index in body["slot_indices"]]

        missing = [
            slot_index
            for slot_index in accepted_slot_indices
            if slot_index not in self.detections_by_slot_index
        ]
        if missing:
            raise ValueError(f"Alice accepted unknown Bob slots: {missing[:5]}")

        self.sifted_slot_indices = accepted_slot_indices
        self.sifted_signal_ids = [
            self.detections_by_slot_index[slot_index].signal_id
            for slot_index in accepted_slot_indices
        ]
        self.sifted_bits = [
            self.detections_by_slot_index[slot_index].bit
            for slot_index in accepted_slot_indices
        ]
        self.reconciled_bits = self.sifted_bits.copy()
        self.sifting_complete = True

    def _on_estimate_sample(self, message, ctx) -> None:
        if not self.sifting_complete:
            raise RuntimeError("cannot estimate QBER before sifting completes")

        body = decode_body(message)
        sample_positions = [int(pos) for pos in body["sample_positions"]]
        sample_bits = [int(bit) for bit in body["sample_bits"]]

        require_positions(
            "sample_positions",
            sample_positions,
            len(self.sifted_bits),
        )
        require_binary_bits("sample_bits", sample_bits)

        if len(sample_positions) != len(sample_bits):
            raise ValueError("sample_positions and sample_bits length mismatch")

        if not sample_positions:
            raise ValueError("sample_positions must not be empty")

        sample_errors = sum(
            self.sifted_bits[position] != alice_bit
            for position, alice_bit in zip(sample_positions, sample_bits)
        )
        qber = sample_errors / len(sample_positions)
        accept = qber <= qber_config.abort_threshold

        self.sample_positions = sample_positions
        self.sample_errors = sample_errors
        self.estimated_qber = qber
        self.qber_accepted = accept
        self.reconciled_bits = remove_positions(
            self.sifted_bits,
            sample_positions,
        )
        self.qber_complete = True

        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="estimate.result",
            body={
                "qber": qber,
                "accept": accept,
                "sample_positions": sample_positions,
                "sample_size": len(sample_positions),
                "sample_errors": sample_errors,
            },
        )

        if not accept:
            self.aborted_reason = "estimated QBER exceeds abort threshold"
            return

        self._start_cascade(ctx)

    def _schedule_local_action(
        self,
        ctx,
        action: str,
        *,
        delay_ticks: int = 0,
    ) -> None:
        ctx.timeline.schedule(
            Event(
                time=ctx.timeline.current_time + delay_ticks,
                target_ref=self,
                action=AGENT_EVENT,
                payload_ref={"bb84_action": action},
                source=self,
                subsystem_id="control",
                meta={
                    "session_id": ctx.session_id,
                    "agent_id": self.agent_id,
                    "bb84_action": action,
                },
            )
        )

    def _schedule_cascade_next(self, ctx) -> None:
        self._schedule_local_action(ctx, "cascade.next_request")

    def _start_cascade(self, ctx) -> None:
        if not self.qber_complete:
            raise RuntimeError("cannot start Cascade before QBER estimation completes")

        if not self.qber_accepted:
            raise RuntimeError("cannot start Cascade after QBER abort")

        n = len(self.reconciled_bits)
        if n == 0:
            raise RuntimeError("cannot start Cascade with empty reconciled bits")

        if self._cascade_rng is None:
            raise RuntimeError("Bob Cascade RNG is not bound")

        block_qber = max(
            float(self.estimated_qber or 0.0),
            cascade_config.block_qber_floor,
            1.0 / n,
        )

        first_block_size = min(
            n,
            cascade_config.max_first_block_size,
            max(cascade_config.min_block_size, math.ceil(0.73 / block_qber)),
        )

        self.cascade_first_block_size = first_block_size
        self._cascade_controller = CascadeController(
            bits=self.reconciled_bits,
            rng=self._cascade_rng,
            passes=cascade_config.passes,
            first_block_size=first_block_size,
        )
        self._schedule_cascade_next(ctx)

    def _cascade_next_request(self, ctx) -> None:
        cascade = self._require_cascade_controller()
        request = cascade.next_request()

        if request is None:
            if cascade.complete:
                self.cascade_complete = True
                self.cascade_parity_requests = cascade.parity_requests
                self.cascade_corrections = cascade.corrections
                self.cascade_leaked_bits = cascade.leaked_bits
                self._start_verification(ctx)
                return

            self._schedule_cascade_next(ctx)
            return

        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="cascade.parity_request",
            body=request.as_body(),
        )

    def _on_cascade_parity_response(self, message, ctx) -> None:
        body = decode_body(message)
        cascade = self._require_cascade_controller()
        cascade.apply_parity_response(
            request_id=str(body["request_id"]),
            alice_parity=int(body["parity"]),
        )
        self._schedule_cascade_next(ctx)

    def _require_cascade_controller(self) -> CascadeController:
        if self._cascade_controller is None:
            raise RuntimeError("Cascade controller is not initialized")
        return self._cascade_controller

    def _start_verification(self, ctx) -> None:
        if not self.cascade_complete:
            raise RuntimeError("cannot verify before Cascade completes")

        if self._verification_rng is None:
            raise RuntimeError("Bob verification RNG is not bound")

        input_len = len(self.reconciled_bits)
        if input_len == 0:
            raise RuntimeError("cannot verify an empty key")

        tag_len = verification_config.tag_len
        tag_seed = random_bits(self._verification_rng, input_len + tag_len - 1)
        tag = toeplitz_hash(self.reconciled_bits, tag_seed, tag_len)

        self.verification_tag_len = tag_len
        self.verification_leaked_bits = tag_len

        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="verify.tag",
            body={
                "hash_family": "toeplitz",
                "input_len": input_len,
                "tag_len": tag_len,
                "tag_seed": tag_seed,
                "tag": tag,
            },
        )

    def _on_verify_result(self, message, ctx) -> None:
        body = decode_body(message)
        verified = body["verified"]

        if not isinstance(verified, bool):
            raise TypeError("verification result must be boolean")

        self.verification_complete = True
        self.verification_accepted = verified

        if not verified:
            self.aborted_reason = "verification failed"
            return

        self._start_privacy_amplification(ctx)

    def _start_privacy_amplification(self, ctx) -> None:
        if not self.verification_accepted:
            raise RuntimeError("cannot amplify privacy before verification passes")

        if self._privacy_rng is None:
            raise RuntimeError("Bob privacy amplification RNG is not bound")

        reconciled_len = len(self.reconciled_bits)

        revealed_bits = (
            len(self.sample_positions)
            + self.cascade_leaked_bits
            + self.verification_leaked_bits
        )

        final_key_len = bb84_final_key_length(
            reconciled_len=reconciled_len,
            estimated_qber=float(self.estimated_qber or 0.0),
            revealed_bits=revealed_bits,
            qber_safety_margin=privacy_config.qber_safety_margin,
            security_margin_bits=privacy_config.security_margin_bits,
        )

        if final_key_len < privacy_config.min_final_key_bits:
            self.aborted_reason = "final key too short"
            return

        seed = random_bits(
            self._privacy_rng,
            reconciled_len + final_key_len - 1,
        )

        self.final_key = toeplitz_hash(
            self.reconciled_bits,
            seed,
            final_key_len,
        )
        self.final_key_length = len(self.final_key)
        self.privacy_revealed_bits = revealed_bits

        send_json_message(
            self,
            ctx,
            receiver_id=self.peer_id,
            out_port=self.out_port,
            message_type="privacy.seed",
            body={
                "input_len": reconciled_len,
                "final_key_len": final_key_len,
                "seed": seed,
                "revealed_bits": revealed_bits,
                "estimated_qber": self.estimated_qber,
                "qber_safety_margin": privacy_config.qber_safety_margin,
                "security_margin_bits": privacy_config.security_margin_bits,
            },
        )


    def _on_privacy_done(self, message, ctx) -> None:
        del ctx

        body = decode_body(message)
        alice_final_key_len = int(body["final_key_len"])

        if alice_final_key_len != self.final_key_length:
            self.aborted_reason = "privacy amplification length mismatch"
            return

        self.privacy_complete = True


# trial runner


In [ ]:
def run_bb84_trial(
    *,
    distance_km: float = 25.0,
    master_seed: int = 2026,
    num_slots: int = 20,
    emission_probability: float | None = None,
    depolarizing_probability: float | None = None,
    detector_efficiency: float | None = None,
    log_file: str | None = None,
) -> dict:
    session_id = f"bb84_distance_{distance_km:g}_seed_{master_seed}"

    source_kwargs = {"num_slots": num_slots}
    if emission_probability is not None:
        source_kwargs["emission_probability"] = emission_probability

    quantum_kwargs = {"length_m": distance_km * 1000}
    if depolarizing_probability is not None:
        quantum_kwargs["depolarizing_probability"] = depolarizing_probability

    detector_kwargs = {}
    if detector_efficiency is not None:
        detector_kwargs["efficiency"] = detector_efficiency

    source_config = BB84AliceSourceConfig(**source_kwargs)
    quantum_config = BB84QuantumChannelConfig(**quantum_kwargs)
    classical_config = BB84ClassicalChannelConfig(length_m=distance_km * 1000)
    detector_config = BB84BobDetectorConfig(**detector_kwargs)

    sampler = StateSampler(
        states=BB84_STATES,
        probabilities=(0.25, 0.25, 0.25, 0.25),
        rep="ket",
        labels=BB84_LABELS,
    )

    source = SinglePhotonSource(
        device_id=source_config.device_id,
        frequency_hz=source_config.clock_hz,
        encoding_scheme=EncodingScheme.POLARIZATION,
        emission_probability=source_config.emission_probability,
        wavelength_nm=source_config.wavelength_nm,
        duration_s=source_config.num_slots / source_config.clock_hz,
        sampler=sampler,
        timing_profile=GaussianTiming(
            mean_emission_delay_ticks=0.0,
            emission_delay_stddev_ticks=seconds_to_ticks(
                source_config.timing_jitter_stddev_s
            ),
            max_emission_delay_ticks=seconds_to_ticks(
                source_config.max_timing_jitter_s
            ),
        ),
    )

    quantum_channel = QuantumChannel(
        channel_id=quantum_config.channel_id,
        length_m=quantum_config.length_m,
        propagation_speed_m_per_s=quantum_config.propagation_speed_m_per_s,
        attenuation_db_per_km=quantum_config.attenuation_db_per_km,
        fixed_insertion_loss_db=quantum_config.fixed_insertion_loss_db,
        timing_jitter_stddev_ticks=seconds_to_ticks(
            quantum_config.timing_jitter_stddev_s
        ),
        noise_models=(
            depolarizing(quantum_config.depolarizing_probability),
        ),
    )

    alice_to_bob_classical = ClassicalChannel(
        channel_id="alice_to_bob_classical",
        length_m=classical_config.length_m,
        fiber_speed_m_per_s=classical_config.fiber_speed_m_per_s,
        loss_probability=classical_config.loss_probability,
        session_id=session_id,
    )

    bob_to_alice_classical = ClassicalChannel(
        channel_id="bob_to_alice_classical",
        length_m=classical_config.length_m,
        fiber_speed_m_per_s=classical_config.fiber_speed_m_per_s,
        loss_probability=classical_config.loss_probability,
        session_id=session_id,
    )

    detector_params = SinglePhotonDetectorParams(
        efficiency=detector_config.efficiency,
        dark_count_rate_hz=detector_config.dark_count_rate_hz,
        dead_time_ticks=seconds_to_ticks(detector_config.dead_time_s),
        jitter_stddev_ticks=seconds_to_ticks(detector_config.jitter_stddev_s),
        p_afterpulse=detector_config.p_afterpulse,
        afterpulse_decay_ticks=seconds_to_ticks(
            detector_config.afterpulse_decay_s
        ),
        photon_number_resolving=False,
    )

    detectors = (
        SinglePhotonDetector("bob_D_Z0", params=detector_params),
        SinglePhotonDetector("bob_D_Z1", params=detector_params),
        SinglePhotonDetector("bob_D_X0", params=detector_params),
        SinglePhotonDetector("bob_D_X1", params=detector_params),
    )

    detector = DetectorArray(
        device_id=detector_config.device_id,
        detectors=detectors,
        measurement=Measure.random(
            (
                (Measure.basis("z", label="Z"), 0.5),
                (Measure.basis("x", label="X"), 0.5),
            ),
            label="bob_random_bb84_basis",
        ),
        readout={
            "Z": {"0": "bob_D_Z0", "1": "bob_D_Z1"},
            "X": {"+": "bob_D_X0", "-": "bob_D_X1"},
        },
        click_resolver=ThresholdClickResolver(double_click_policy="random"),
        detection_window_ticks=seconds_to_ticks(
            detector_config.detection_window_s
        ),
        consume_signal=True,
        output_latency_ticks=seconds_to_ticks(
            detector_config.output_latency_s
        ),
    )

    alice = BB84AliceAgent(
        agent_id="alice_agent",
        node_id="alice",
        source=source,
        quantum_done_delay_ticks=bb84_source_done_delay_ticks(source_config),
    )

    bob = BB84BobAgent(
        agent_id="bob_agent",
        node_id="bob",
        sifting_guard_ticks=bb84_bob_sifting_guard_ticks(detector_config),
        slot_period_ticks=bb84_slot_period_ticks(source_config),
        slot_zero_arrival_tick=bb84_slot_zero_arrival_tick(quantum_config),
        slot_assignment_window_ticks=bb84_slot_assignment_window_ticks(
            source_config,
            quantum_config,
            detector_config,
        ),
        num_slots=source_config.num_slots,
    )

    network = Network(topology_id="bb84_two_node_network")

    alice_node = Node("alice")
    bob_node = Node("bob")

    alice_classical = alice.enable_classical()
    bob_classical = bob.enable_classical()

    alice_classical.add_route("bob_agent", "to_bob")
    bob_classical.add_route("alice_agent", "to_alice")

    alice_node.add_device("source", source)
    alice_node.add_agent(alice)

    bob_node.add_device("detector", detector)
    bob_node.add_agent(bob)

    network.add_node(alice_node)
    network.add_node(bob_node)

    network.add_quantum_link(
        "alice_to_bob_fiber",
        "alice",
        "bob",
        channel=quantum_channel,
    )

    network.add_classical_link(
        "alice_to_bob_public",
        "alice",
        "bob",
        channel=alice_to_bob_classical,
    )

    network.add_classical_link(
        "bob_to_alice_public",
        "bob",
        "alice",
        channel=bob_to_alice_classical,
    )

    network.wire_ports(
        "alice_source_to_quantum_channel",
        source.output_port,
        quantum_channel.input_port,
        target_action=ACTION_TRANSMIT_QUANTUM,
    )

    network.wire_ports(
        "quantum_channel_to_bob_detector",
        quantum_channel.output_port,
        detector.input_port,
        target_action=ACTION_DETECT_SIGNAL,
    )

    network.wire_ports(
        "alice_source_report_to_alice_agent",
        source.report_port,
        alice.report_port,
        target_action=AGENT_REPORT,
    )

    network.wire_ports(
        "bob_detector_report_to_bob_agent",
        detector.output_port,
        bob.report_port,
        target_action=AGENT_REPORT,
    )

    network.wire_ports(
        "alice_agent_to_alice_bob_classical_channel",
        alice_classical.out_port("to_bob"),
        alice_to_bob_classical.input_port,
        target_action=ACTION_TRANSMIT_CLASSICAL,
    )

    network.wire_ports(
        "alice_bob_classical_channel_to_bob_agent",
        alice_to_bob_classical.output_port,
        bob_classical.in_port("from_alice"),
        target_action=AGENT_MESSAGE,
    )

    network.wire_ports(
        "bob_agent_to_bob_alice_classical_channel",
        bob_classical.out_port("to_alice"),
        bob_to_alice_classical.input_port,
        target_action=ACTION_TRANSMIT_CLASSICAL,
    )

    network.wire_ports(
        "bob_alice_classical_channel_to_alice_agent",
        bob_to_alice_classical.output_port,
        alice_classical.in_port("from_bob"),
        target_action=AGENT_MESSAGE,
    )

    sinks = []
    if log_file is not None:
        sinks.append(
            JsonlSink(
                path=Path(log_file),
                session_id=session_id,
                auto_flush=True,
                append=False,
            )
        )

    logger = SimulationLogger(
        level=LogLevel.DEBUG,
        sinks=sinks,
        session_id=session_id,
    )

    timeline = Timeline(
        master_seed=master_seed,
        logger=logger,
    )

    runtime = SessionRuntime(
        timeline=timeline,
        network=network,
        session_id=session_id,
        start_time=0,
    )

    runtime.bind_all()
    runtime.schedule_agent_starts()

    runtime.run_until_empty()

    for sink in sinks:
        sink.flush()

    length_km = quantum_config.length_m / 1000
    channel_loss_db = quantum_config.attenuation_db_per_km * length_km
    total_optical_loss_db = channel_loss_db + quantum_config.fixed_insertion_loss_db
    channel_transmission = 10 ** (-total_optical_loss_db / 10)

    return {
        "distance_km": distance_km,
        "master_seed": master_seed,
        "num_slots": num_slots,
        "emission_probability": source_config.emission_probability,
        "depolarizing_probability": quantum_config.depolarizing_probability,
        "detector_efficiency": detector_config.efficiency,
        "prepared_photons": len(alice.preparations),
        "emitted_slots": len(alice.preparations_by_slot_index),
        "no_emission_slots": num_slots - len(alice.preparations_by_slot_index),
        "channel_received": quantum_channel.received_count,
        "channel_delivered": quantum_channel.delivered_count,
        "channel_lost": quantum_channel.lost_count,
        "bob_reports": len(bob.reports),
        "bob_detections": len(bob.detections),
        "bob_detected_slots": len(bob.detections_by_slot_index),
        "bob_no_usable_detection_slots": num_slots - len(bob.detections_by_slot_index),
        "bob_failed_reports": len(bob.failed_reports),
        "bob_unassigned_reports": len(bob.unassigned_reports),
        "bob_duplicate_slot_reports": len(bob.duplicate_slot_reports),
        "alice_no_emission_sift_rejections": alice.no_emission_sift_rejections,
        "sifted_bits": len(bob.sifted_bits),
        "sifted_slots": len(bob.sifted_slot_indices),
        "sample_size": len(bob.sample_positions),
        "sample_errors": bob.sample_errors,
        "estimated_qber": bob.estimated_qber,
        "qber_accepted": bob.qber_accepted,
        "cascade_complete": bob.cascade_complete,
        "cascade_parity_requests": bob.cascade_parity_requests,
        "cascade_corrections": bob.cascade_corrections,
        "cascade_leaked_bits": bob.cascade_leaked_bits,
        "reconciled_key_length": len(bob.reconciled_bits),
        "reconciled_bits_equal": alice.reconciled_bits == bob.reconciled_bits,
        "verification_accepted": bob.verification_accepted,
        "verification_leaked_bits": bob.verification_leaked_bits,
        "privacy_complete": bob.privacy_complete,
        "privacy_revealed_bits": bob.privacy_revealed_bits,
        "final_key_length": bob.final_key_length,
        "final_keys_equal": (
            bob.privacy_complete and alice.final_key == bob.final_key
        ),
        "protocol_complete": (
            bob.privacy_complete and alice.final_key == bob.final_key
        ),
        "alice_abort": alice.aborted_reason,
        "bob_abort": bob.aborted_reason,
        "channel_transmission": channel_transmission,
        "expected_detection_rate_per_prepared": (
            channel_transmission * detector_config.efficiency
        ),
        "observed_detection_rate_per_prepared": (
            len(bob.detections) / len(alice.preparations)
            if alice.preparations
            else 0.0
        ),
        "sifting_fraction": (
            len(bob.sifted_bits) / len(bob.detections)
            if bob.detections
            else 0.0
        ),
    }


## Tiny Physical Scenario

Parameters used below:

- 5 km fiber: real propagation delay and optical loss, but short enough that Bob still detects several photons.
- 20 source slots with emission probability 1.0: exactly 20 attempted emissions, so the trace is as small as possible while still showing multiple BB84 slots.
- Detector efficiency 0.85: some photons can arrive but fail detection.
- Depolarizing probability 0.02: small non-ideal channel noise.

This run is mainly for quantum transmission and sifting. With so few sifted bits, QBER/Cascade/privacy may abort, and that is okay for this teaching notebook.


In [ ]:
def _meta_dict(record: dict) -> dict:
    return {key: value for key, value in record.get("meta", [])}


def _bb84_label_text(label: str | None) -> str:
    mapping = {
        "Z0": "Z basis, bit 0",
        "Z1": "Z basis, bit 1",
        "X0": "X basis, bit 0 (+)",
        "X1": "X basis, bit 1 (-)",
    }
    return mapping.get(label, str(label))


def _bob_outcome_bit(measurement_label: str | None, outcome: str | None) -> int | None:
    if measurement_label == "Z" and outcome in {"0", "1"}:
        return int(outcome)
    if measurement_label == "X" and outcome == "+":
        return 0
    if measurement_label == "X" and outcome == "-":
        return 1
    return None


def load_bb84_trace_records(path: str | Path) -> list[dict]:
    path = Path(path)
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def build_bb84_teaching_trace(path: str | Path) -> list[dict]:
    records = load_bb84_trace_records(path)
    emitted_by_signal_id = {}
    lines = []

    for record in records:
        meta = _meta_dict(record)
        category = record.get("category")
        time = record.get("sim_time")

        if category == "components.sources.single_photon_source.emit":
            signal_id = meta.get("signal_id")
            slot_index = int(meta.get("attempt_index", 0)) - 1
            label = meta.get("sampler_label")
            emitted_by_signal_id[signal_id] = {
                "slot_index": slot_index,
                "label": label,
            }
            lines.append({
                "time": time,
                "kind": "emit",
                "text": (
                    f"Alice emitted photon in slot {slot_index}: "
                    f"{_bb84_label_text(label)}"
                ),
            })

        elif category == "components.channels.quantum.signal_lost":
            signal_id = meta.get("signal_id")
            emitted = emitted_by_signal_id.get(signal_id, {})
            slot_index = emitted.get("slot_index", "?")
            lines.append({
                "time": time,
                "kind": "loss",
                "text": f"Photon from slot {slot_index} was lost in the quantum channel",
            })

        elif category == "components.channels.quantum.signal_forwarded":
            signal_id = meta.get("signal_id")
            emitted = emitted_by_signal_id.get(signal_id, {})
            slot_index = emitted.get("slot_index", "?")
            arrival_time = meta.get("arrival_time")
            lines.append({
                "time": time,
                "kind": "forward",
                "text": (
                    f"Photon from slot {slot_index} entered the fiber; "
                    f"scheduled to reach Bob at t={arrival_time}"
                ),
            })

        elif category == "components.detectors.detector_array.detect":
            signal_id = meta.get("signal_id")
            emitted = emitted_by_signal_id.get(signal_id, {})
            slot_index = emitted.get("slot_index", "?")
            basis = meta.get("measurement_label")
            outcome = meta.get("outcome")
            bit = _bob_outcome_bit(basis, outcome)
            success = meta.get("success")
            if success:
                lines.append({
                    "time": time,
                    "kind": "detect",
                    "text": (
                        f"Bob detected slot {slot_index}: measured in {basis}, "
                        f"outcome {outcome}, bit {bit}"
                    ),
                })
            else:
                lines.append({
                    "time": time,
                    "kind": "miss",
                    "text": f"Bob received slot {slot_index}, but detector produced no usable click",
                })

        elif category == "components.channels.classical.message_forwarded":
            message_type = meta.get("message_type")
            arrival_time = meta.get("arrival_time")
            if message_type == "quantum.done":
                text = f"Alice sent `quantum.done` to Bob; scheduled arrival t={arrival_time}"
            elif message_type == "sift.bob_bases":
                text = f"Bob sent detected-slot bases to Alice; scheduled arrival t={arrival_time}"
            elif message_type == "sift.accepted":
                text = f"Alice sent accepted matching-basis slots to Bob; scheduled arrival t={arrival_time}"
            elif message_type == "abort":
                # The tiny teaching run may abort later because post-processing
                # needs more sifted bits. This trace focuses on quantum + sifting.
                continue
            else:
                continue
            lines.append({
                "time": time,
                "kind": "classical",
                "text": text,
            })

    return sorted(lines, key=lambda item: (item["time"], item["kind"], item["text"]))


def show_bb84_teaching_trace(path: str | Path, *, limit: int | None = None) -> None:
    trace = build_bb84_teaching_trace(path)
    if limit is not None:
        trace = trace[:limit]
    for item in trace:
        print(f"t={item['time']:>10} | {item['text']}")


In [ ]:
teaching_log_path = PROTOCOLS_DIR / "bb84_teaching_trace.jsonl"

teaching_trial = run_bb84_trial(
    distance_km=0.005,
    master_seed=77,
    num_slots=20,
    emission_probability=1.0,
    depolarizing_probability=0.02,
    detector_efficiency=0.85,
    log_file=str(teaching_log_path),
)

important_summary = {
    "num_slots": teaching_trial["num_slots"],
    "prepared_photons": teaching_trial["prepared_photons"],
    "no_emission_slots": teaching_trial["no_emission_slots"],
    "channel_delivered": teaching_trial["channel_delivered"],
    "channel_lost": teaching_trial["channel_lost"],
    "bob_detections": teaching_trial["bob_detections"],
    "bob_failed_reports": teaching_trial["bob_failed_reports"],
    "sifted_bits": teaching_trial["sifted_bits"],
}

print(json.dumps(important_summary, indent=2))
print("Saved teaching log to:", teaching_log_path)


In [ ]:
show_bb84_teaching_trace(teaching_log_path)


## How To Read The Trace

- `Alice emitted` means the source actually prepared a photon for that slot.
- `lost in the quantum channel` means the photon did not reach Bob.
- `entered the fiber` means the channel scheduled an arrival at Bob after propagation delay and jitter.
- `Bob detected` means the detector produced a usable measurement result.
- `quantum.done` is Alice's classical message telling Bob the source frame is finished.
- `sift.bob_bases` is Bob announcing only the bases for the slots he detected.
- `sift.accepted` is Alice telling Bob which detected slots had matching bases.
